# Analysis of networks of persons in relation to organisations


Cf. [this notebook](https://github.com/Sciences-historiques-numeriques/histoire_numerique_methodes/blob/main/analyse_reseaux/reseaux_florence.ipynb) about families and power in Renaissance Florence for an introduction to network analysis with the *networkx* Python library

In [ ]:
import pandas as pd


import networkx as nx
from networkx.algorithms import bipartite

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.ticker import MaxNLocator

from scipy.stats import spearmanr
import statsmodels.api as sm

import numpy as np
import seaborn as sns
import math
import os

In [ ]:
### Librairies déjà installées avec Python
import pprint
import csv

import sqlite3 as sql

### this library allows to do SQL queries on dataframes
import duckdb

import pickle

import time
import datetime
from dateutil import parser


from shutil import copyfile


In [ ]:
### Importing a custom function module
##  NOTE: The ‘sparql_functions.py’ file must be located 
#   in a folder that is included in the search path
#   recognized by this Jupyter notebook so that
#   the import works correctly

import sys
from importlib import reload

# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')



In [ ]:
import network_analysis_functions as naf
import bivariate_library as bl

In [ ]:
print(reload(bl)) 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Get and inspect the data


### Get the network data

We use here the network data prepared in this notebook 
* Get the [Network creation](https://github.com/CamillaDM/war_correspondents_2/blob/main/notebooks_jupyter/wikidata_exploration/da6-persons-network-creation.ipynb).

In [ ]:
### Upload the network prepared in the Network creation

file_address='../da_data/da6-networks.pkl'
network=pd.read_pickle(file_address)
print(len(network))
network.iloc[:3]

#### Swap the values in order to have always older person in position x

We want to have the older person always in the in the first position.

This will be useful when coloring the network's edges: you then have to choose between one of the two persons and this gives the direction of the flow of time

In [ ]:
### Get the rows where persons chronologically inversed
inverse_df=network[network['per_activ_x']>network['per_activ_y']]
print(len(inverse_df))
inverse_df.head()

In [ ]:
### Copy networks to work on copies
df = network.copy()
df_new = network.copy()

In [ ]:
### Swap only the rosw in index


# The specific indices where the swap must occur
indices_to_swap = inverse_df.index


# Identify all column pairs (_x and _y) dynamically
x_cols = [col for col in df.columns if col.endswith('_x')]
pairs = []
for x_col in x_cols:
    base = x_col.replace('_x', '')
    y_col = f"{base}_y"
    if y_col in df.columns:
        pairs.append((x_col, y_col))

# Perform the swap ONLY for the rows in indices_to_swap
for x_col, y_col in pairs:
    # Extract values at the specific indices
    vals_x = df_new.loc[indices_to_swap, x_col].values
    vals_y = df_new.loc[indices_to_swap, y_col].values
    
    # Assign swapped values back to the same specific indices
    df_new.loc[indices_to_swap, x_col] = vals_y
    df_new.loc[indices_to_swap, y_col] = vals_x

# df_new now has swapped values for rows in index
# All other rows remain exactly as they were in the original df

##### Check swap result

In [ ]:
## example
ii=inverse_df.sort_values(by='sum_edu_empl', ascending=False).index
print(ii[10:15])

In [ ]:
df_new.iloc[ii[10:15]]

In [ ]:
network.iloc[ii[10:15]]

In [ ]:
### Replace the original network

network=df_new


### Get the data about persons


#### Get the individuals (800) used for correspondence analysis 


In [ ]:
### Information about persons with gender, birth date,
# coded country, etc.
csv_address='../da_data/da4-AFC.csv'
### csv_address='da_data/da2-birth-place.csv'
df_p = pd.read_csv(csv_address)
df_p=df_p[['uriPer', 'labelPer', 'birthYear', 'gender', 'labelPlace', 'REGION', 'NAME_ENGL', 'coded_country', 'periodsActivity']]
### Rename columns to have shorter labels
df_p.columns=['person_uri',
 'labelPer',
 'birthYear',
 'gender',
 'labelPlace',
 'REGION',
 'NAME_ENGL',
 'country',
 'per_activ'
 ]


In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_p.info()

In [ ]:
print(len(df_p))
df_p.iloc[:3]

## Create the graphs

We will now transform our networks into graphs using the Python *networkx* library

In [ ]:
network.iloc[:2]

### Education

In [ ]:
### Take all the relathioships with an educational link 
ntwk_edu=network[network.orgs_number_edu>0][['labelPer_x', 'labelPer_y', 'birthYear_x', 'birthYear_y', 'per_activ_x', 'per_activ_y', 'sum_edu_empl', 'sum_all_rel', 'orgs_number_edu', 'orgs_labels_edu','person_uri_x', 'person_uri_y']]
print(len(ntwk_edu))
ntwk_edu.iloc[:2]

In [ ]:
## Provide the data in the format 
# required by Networkx

l = [tuple(
    (e['person_uri_x'], e['person_uri_y'],
     {'orgsLabels':e['orgs_labels_edu'], 'orgsNumber':e['orgs_number_edu'],
      'per_activ_x':e['per_activ_x'], 'per_activ_y':e['per_activ_y']}
     )) 
     for e in ntwk_edu.to_dict(orient='records')]
print(len(l))

In [ ]:
## Créate the empty graph
G_edu=nx.Graph()

## Add relationships to graph
# Multiple rows between two edges are taken only once
G_edu.add_edges_from(l)

naf.basic_graph_properties(G_edu)


#### Add metadata to nodes

In [ ]:
df_p.head(2)

In [ ]:
df_pm = df_p[['person_uri','labelPer', 'birthYear', 'gender','per_activ','country']]
df_pm = df_pm.drop_duplicates()
df_pm.columns=['uri', 'label', 'birthYear', 'gender','per_activ','country']
df_pm.head()

In [ ]:
### Prepare data to add to nodes
# The data must be of type : dictionary, i.e. key:value structure
ln = dict([(e['uri'],
     {'label':e['label'], 
       'birthYear':e['birthYear'],
       'gender':e['gender'],
       'per_activ':e['per_activ'],
      'country':e['country']}
     ) for e in df_pm.to_dict(orient='records')])
# print(str(l)[:200])

In [ ]:
## Add attributes
nx.set_node_attributes(G_edu, ln)
pprint.pprint(list(G_edu.nodes.data())[:2])


In [ ]:
naf.basic_graph_properties(G_edu)

#### Edu Components

This graph isn't connected. This means that there are parts of the graph, called 'components', that are separated from other parts.

In [ ]:
### Create a list of graphs, one per component
perS_edu = [G_edu.subgraph(c).copy() for c in nx.connected_components(G_edu)]

### i is the component index in the list S of graphs , len(s.nodes) is the nomber of nodes
lc = sorted([[i,len(s.nodes)] for i,s in enumerate(perS_edu)], key = lambda row: row[1], reverse=True)
print(lc[:5])

### Again we observe that there is a big connected graphe 
# and a multitude of small graphs

### Explore small graph

In [ ]:
### 
li = [1]    # [48,36, ..]
ll = [list(perS_edu[i[0]].nodes.data()) for i in ln if i[0] in li ]
#pprint.pprint(str(ll)[:300])

In [ ]:

# Display variable

variable_name = 'edu_2'

globals()[variable_name] = nx.Graph()

edu_gu=globals()[variable_name]

for i in li:
    ## ajoute au graphe les composantes en utilisant
    # l'index ou position dans la liste de graphes 'S'
    edu_gu = nx.union(edu_gu, perS_edu[i])
naf.basic_graph_properties(edu_gu)


In [ ]:
### Draw graph with networkx



# print(list(g.nodes.data())[:3])
# print(list(g.edges.data())[:3])

n_size = np.log(np.sqrt(nx.number_of_nodes(edu_gu)))*20 #*25

graph_layout = 'kamada_kawai'
n_k = 0.4
sc = 0.02

### Define the layout, i.e. the choice 
# of the algorithm for the representation of the graph.

if graph_layout == 'fruchterman_reingold':
    pos = nx.fruchterman_reingold_layout(edu_gu)
elif graph_layout == 'kamada_kawai':
    pos = nx.kamada_kawai_layout(edu_gu)
elif graph_layout == 'spring_layout':
    pos = nx.spring_layout(g, k = n_k, scale=sc)  
else:
    pos = nx.kamada_kawai_layout(edu_gu)


# https://networkx.org/documentation/stable/reference/drawing.html
plt.figure(figsize = (n_size,n_size))

node_size = [d[1]*n_size for d in nx.degree(edu_gu)]
node_labels = dict([tuple(( n[0] , n[1]['label'] ))for n in edu_gu.nodes.data()])
#print(node_labels)
edge_labels = {e: edu_gu.get_edge_data(e[0], e[1])["orgsLabels"] for e in edu_gu.edges()}
#print(edge_labels)


### On représente successivement les différentes sommets et arêtes,
# puis on ajoute les labels
nx.draw_networkx_nodes(edu_gu, pos,  node_size=node_size, alpha=0.4)
nx.draw_networkx_edges(edu_gu, pos, label=edge_labels, width=4, alpha=0.2) # edgelist=ln, edge_color=c, 
nx.draw_networkx_labels(edu_gu, pos, labels=node_labels, alpha=0.7, font_size=n_size/2)
nx.draw_networkx_edge_labels(edu_gu, pos=pos, edge_labels=edge_labels, alpha=0.6)

### On peut augmenter ou diminuer ce paramètre pour ajuster le graphe
plt.tight_layout(pad=50)
plt.savefig('images/small_bipartite_component.svg')
plt.show()


### Explore with Gephi online

* export the graph in the Gephi format
* explore it uploading the file into the [Gephi Web User Interface](https://lite.gephi.org)
* use the ForceAtlas2 algorythm: first click on the 'magic' button (left of the 'Start' button) then on Start

In [ ]:
### Graph to heavy to use Gephy online
nx.write_gexf(edu_gu, f"da_graphs/{edu_gu}.gexf")

### Main component

In [ ]:
### Get the list of the nodes in the main component
li = [1]    
ll = [list(perS_edu[i[1]].nodes.data()) for i in ln if i[1] in li]

In [ ]:
# Display variable

variable_name = 'edu_2'

globals()[variable_name] = nx.Graph()

edu_gu=globals()[variable_name]

for i in li:
    ## ajoute au graphe les composantes en utilisant
    # l'index ou position dans la liste de graphes 'S'
    edu_gu = nx.union(edu_gu, perS_edu[i])
naf.basic_graph_properties(edu_gu)


In [ ]:
pprint.pprint(list(edu_gu.nodes.data())[2:4])

In [ ]:
### Graph to heavy to use Gephy online
#nx.write_gexf(gu, f"da_graphs/{variable_name}.gexf")

### Employment graph

In [ ]:
### Take all the relathioships with an educational link 
ntwk_empl=network[network.orgs_number_empl>0][['labelPer_x', 'labelPer_y', 'birthYear_x', 'birthYear_y', 'per_activ_x', 'per_activ_y', 'sum_edu_empl', 'sum_all_rel', 'orgs_number_empl', 'orgs_labels_empl','person_uri_x', 'person_uri_y']]
print(len(ntwk_empl))
ntwk_empl.iloc[:2]


In [ ]:
## Provide the data in the format 
# required by Networkx

l = [tuple(
    (e['person_uri_x'], e['person_uri_y'],
     { 'orgsLabels':e['orgs_labels_empl'], 'orgsNumber':e['orgs_number_empl'],   
      'periods_x':e['per_activ_x'], 'periods_y':e['per_activ_y']}
     )) 
     for e in ntwk_empl.to_dict(orient='records')]
print(len(l))

In [ ]:
## Créate the empty graph
G_empl=nx.Graph()

## Add relationships to graph
# Multiple rows between two edges are taken only once
G_empl.add_edges_from(l)

naf.basic_graph_properties(G_empl)


In [ ]:
### Add person attributes to nodes

## the ln list was prepared above

## Add attributes
nx.set_node_attributes(G_empl, ln)
pprint.pprint(list(G_empl.nodes.data())[:2])


In [ ]:
### Create a list of graphs, one per component
perS_empl = [G_empl.subgraph(c).copy() for c in nx.connected_components(G_empl)]

### i is the component index in the list S of graphs , len(s.nodes) is the nomber of nodes
lc = sorted([[i,len(s.nodes)] for i,s in enumerate(perS_empl)], key = lambda row: row[1], reverse=True)
print(lc[:5])

### Again we observe that there is a big connected graphe 
# and a multitude of small graphs

#### Main component

In [ ]:
### 
li = [0]    # [15, 33, 30]
ll = [list(perS_empl[i[0]].nodes.data()) for i in ln if i[0] in li ]
##pprint.pprint(str(ll)[:300])

In [ ]:
# Display variable

variable_name = 'empl_0'

globals()[variable_name] = nx.Graph()

empl_gu=globals()[variable_name]

for i in li:
    ## ajoute au graphe les composantes en utilisant
    # l'index ou position dans la liste de graphes 'S'
    empl_gu = nx.union(empl_gu, perS_empl[i])
naf.basic_graph_properties(empl_gu)

### Membership graph

In [ ]:
### Take all the relathioships with an educational link 
ntwk_memb=network[network.orgs_number_memb>0][['labelPer_x', 'labelPer_y', 'birthYear_x', 
                                               'birthYear_y', 'per_activ_x', 'per_activ_y', 
                                               'sum_edu_empl', 'sum_all_rel', 
                                               'orgs_number_memb', 'orgs_labels_memb','person_uri_x', 'person_uri_y']]
print(len(ntwk_memb))
ntwk_memb.iloc[:2]


In [ ]:
## Provide the data in the format 
# required by Networkx

l = [tuple(
    (e['person_uri_x'], e['person_uri_y'],
     { 'orgsLabels':e['orgs_labels_memb'], 'orgsNumber':e['orgs_number_memb'],   
      'periods_x':e['per_activ_x'], 'periods_y':e['per_activ_y']}
     )) 
     for e in ntwk_memb.to_dict(orient='records')]
print(len(l))

In [ ]:
## Créate the empty graph
G_memb=nx.Graph()

## Add relationships to graph
# Multiple rows between two edges are taken only once
G_memb.add_edges_from(l)

naf.basic_graph_properties(G_memb)


In [ ]:
### Add person attributes to nodes

## the ln list was prepared above

## Add attributes
nx.set_node_attributes(G_memb, ln)
pprint.pprint(list(G_memb.nodes.data())[:2])


In [ ]:
### Create a list of graphs, one per component
perS_memb = [G_memb.subgraph(c).copy() for c in nx.connected_components(G_memb)]

### i is the component index in the list S of graphs , len(s.nodes) is the nomber of nodes
lc = sorted([[i,len(s.nodes)] for i,s in enumerate(perS_memb)], key = lambda row: row[1], reverse=True)
print(lc[:5])

### Again we observe that there is a big connected graphe 
# and a multitude of small graphs

#### Main component

In [ ]:
### 
li = [0]    # [15, 33, 30]
ll = [list(perS_memb[i[0]].nodes.data()) for i in ln if i[0] in li ]
#pprint.pprint(str(ll)[:300])

In [ ]:

# Display variable

variable_name = 'memb_0'

globals()[variable_name] = nx.Graph()

memb_gu=globals()[variable_name]

for i in li:
    ## ajoute au graphe les composantes en utilisant
    # l'index ou position dans la liste de graphes 'S'
    memb_gu = nx.union(memb_gu, perS_memb[i])
naf.basic_graph_properties(memb_gu)


## Multylayer analysis : position of persons

The idea and code of this analysis comes from prompting Euria in May 2026: [here is the link](https://euria.infomaniak.com/shared/019e34b4-1ca7-73d6-841d-075a744e609b)





Analysing the positions of physicists across three distinct relational layers (education, employment and academy membership) constitutes an analysis of a multiplex network (or multilayer network). Each graph represents a different phase or dimension of the physicists' career:
1.  **Education Graph:** Peer formation and early networking (latent potential).
2.  **Employment Graph:** Professional collaboration and institutional power (active production).
3.  **Membership Graph:** Prestige, recognition, and gatekeeping (established authority).

### 1. The Career Trajectory
Instead of looking at the graphs in isolation, map the evolution of an actor's centrality:
*   **The "Star" Trajectory:** High degree in Education $\rightarrow$ High degree in Employment $\rightarrow$ High degree in Membership. (Consistent elite).
*   **The "Hidden Gem":** Low/Medium in Education $\rightarrow$ High in Employment $\rightarrow$ High in Membership. (Rose through merit/work).
*   **The "Academic Prince":** High in Education (popular student) $\rightarrow$ Low in Employment $\rightarrow$ High in Membership. (Recognized for potential/pedigree rather than output).
*   **The "Workhorse":** Low in Education $\rightarrow$ High in Employment $\rightarrow$ Low in Membership. (Productive but excluded from elite academies).

### 2. Method: Multilayer Centrality Correlation
Calculate specific centrality measures for each graph and correlate them to see if "power" transfers between layers.

**Recommended Metrics per Layer:**
*   **Education:**  *Degree Centrality* (or alternatively *Clustering Coefficient* about tightness of student cohorts)
*   **Employment:** *Betweenness Centrality* (bridging different departments/universities).
*   **Membership:** *Eigenvector Centrality* (working with other important people).


### Prepare sets

In [ ]:
print(edu_gu.number_of_nodes())
print(empl_gu.number_of_nodes())
print(memb_gu.number_of_nodes())
all_nodes=set(edu_gu.nodes()) & set(empl_gu.nodes())  & set(memb_gu.nodes())
print(len(all_nodes))


In [ ]:
### Creates aligned subgraphs containing only nodes present in ALL three layers.
    

def prepare_multilayer_core(G_edu_full, G_emp_full, G_mem_full):
    
    
    # 1. Find the Intersection of Nodes
    # This gives us the list of war-journalists with complete data across all 3 layers
    common_nodes = set(G_edu_full.nodes()) & set(G_emp_full.nodes()) & set(G_mem_full.nodes())
    
    print(f"Original counts: Edu={G_edu_full.number_of_nodes()}, Emp={G_emp_full.number_of_nodes()}, Mem={G_mem_full.number_of_nodes()}")
    print(f"Core intersection size: {len(common_nodes)} war-journalists")
    
    if len(common_nodes) == 0:
        raise ValueError("No common nodes found across the three graphs. Check your node IDs.")

    # 2. Create Subgraphs
    # We use .copy() to create independent graph objects.
    # This is crucial: it removes all edges connected to nodes NOT in common_nodes.
    G_edu_core = G_edu_full.subgraph(common_nodes).copy()
    G_emp_core = G_emp_full.subgraph(common_nodes).copy()
    G_mem_core = G_mem_full.subgraph(common_nodes).copy()
    
    # 3. Verification (Optional but recommended)
    assert set(G_edu_core.nodes()) == set(G_emp_core.nodes()) == set(G_mem_core.nodes())
    
    return G_edu_core, G_emp_core, G_mem_core


In [ ]:
# Create the subgraphs
GA_edu, GA_empl, GA_memb = prepare_multilayer_core(edu_gu, empl_gu, memb_gu)


In [ ]:
# Test des intersections deux à deux
inter_edu_empl = set(edu_gu.nodes) & set(empl_gu.nodes)
inter_empl_memb = set(empl_gu.nodes) & set(memb_gu.nodes)
inter_edu_memb = set(edu_gu.nodes) & set(memb_gu.nodes)

print(f"Intersection Éducation/Emploi : {len(inter_edu_empl)} journalistes")
print(f"Intersection Emploi/Clubs     : {len(inter_empl_memb)} journalistes")
print(f"Intersection Éducation/Clubs  : {len(inter_edu_memb)} journalistes")

In [ ]:
# Afficher les nœuds du réseau des Clubs
print("Journalistes dans Clubs :", list(memb_gu.nodes))

# Afficher les nœuds du réseau Éducation
print("Journalistes dans Éducation :", list(edu_gu.nodes))

# Afficher les nœuds du réseau Emploi
print("Journalistes dans Emploi :", list(empl_gu.nodes))

### Inspect sets

In [ ]:
naf.basic_graph_properties(GA_edu)

In [ ]:
naf.basic_graph_properties(GA_empl)

In [ ]:
naf.basic_graph_properties(GA_memb)

In [ ]:
# --- Helper Function Defined at Top Level ---
def classify_actor(row, high_thresh=0.8, low_thresh=-0.2):
    """
    Classifies a physicist based on standardized scores (Z-scores).
    Expects a pandas Series with 'edu_z', 'emp_z', 'mem_z'.
    """
    edu, emp, mem = row['edu_z'], row['emp_z'], row['mem_z']
    
    if edu > high_thresh and emp > high_thresh and mem > high_thresh:
        return "Star"
    if edu < low_thresh and emp > high_thresh and mem > high_thresh:
        return "Hidden Gem"
    if emp > high_thresh and mem < low_thresh:
        return "Workhorse"
    if edu > high_thresh and emp < low_thresh and mem > high_thresh:
        return "Academic Prince"
    return "Standard"

def analyze_career_trajectories(G_edu, G_emp, G_mem):
    """
    Main analysis function.
    """
    # 1. Align Nodes
    common_nodes = set(G_edu.nodes()) & set(G_emp.nodes()) & set(G_mem.nodes())
    if not common_nodes:
        raise ValueError("No common nodes found.")
    
    # 2. Calculate Metrics
    edu_scores = nx.degree_centrality(G_edu)
    emp_scores = nx.betweenness_centrality(G_emp)
    mem_scores = nx.eigenvector_centrality(G_mem, max_iter=1000)
    
    # 3. Create DataFrame
    df = pd.DataFrame({
        'physicist': list(common_nodes),
        'edu_raw': [edu_scores[n] for n in common_nodes],
        'emp_raw': [emp_scores[n] for n in common_nodes],
        'mem_raw': [mem_scores[n] for n in common_nodes]
    })
    
    # 4. Standardize
    cols = ['edu_raw', 'emp_raw', 'mem_raw']
    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()
    df.rename(columns={'edu_raw': 'edu_z', 'emp_raw': 'emp_z', 'mem_raw': 'mem_z'}, inplace=True)
    
    # 5. Apply Classification
    # We pass the function object to apply(). 
    # 'axis=1' means the function receives one row at a time.
    df['role'] = df.apply(classify_actor, axis=1)
    
    # 6. Return Results
    return {
        'dataframe': df,
        'stars': df[df['role'] == 'Star']['physicist'].tolist(),
        'hidden_gems': df[df['role'] == 'Hidden Gem']['physicist'].tolist(),
        'role_counts': df['role'].value_counts()
    }

In [ ]:
### Create Multilayer Centrality Correlation

# This can take some minutes

results = analyze_career_trajectories(GA_edu, GA_empl, G_memb)

# print(f"Stars identified: {len(results['stars'])}")
# print(results['stars'])
# print(f"\nHidden Gems identified: {len(results['hidden_gems'])}")
# print(results['hidden_gems'])
# print("\nCorrelation between layers:")
# print(results['correlation_matrix'])
# print("\nRole Distribution:")
# print(results['role_counts'])

In [ ]:
results_dataframe=results['dataframe']
print(len(results_dataframe))
results_dataframe.rename(columns={'physicist':'uri'}, inplace=True)
results_dataframe.iloc[:3]

In [ ]:
print(results_dataframe.groupby(by='role').size().sort_values(ascending=False))

In [ ]:
df_pa=pd.merge(df_pm,results_dataframe,left_on='uri',right_on='uri')

In [ ]:
df_pa.iloc[:3]

In [ ]:
duckdb.query(
    f"""
    SELECT * 
    from df_pa 
    where label like '%Einstein%'
    or label like '%Planck%'
    """
).to_df()



In [ ]:
duckdb.query(
    f"""
    SELECT * 
    from df_pa 
    --where role = 'Hidden Gem'
    --where role = 'Star'
    where role = 'Academic Prince'
    LIMIT 10
    """
).to_df()


In [ ]:
def visualize_role_structures(df):
    """
    df must contain: 'role', 'edu_z', 'emp_z', 'mem_z'
    """
    # Melt the dataframe to long format for easy plotting
    plot_data = df.melt(id_vars=['role'], 
                        value_vars=['edu_z', 'emp_z', 'mem_z'],
                        var_name='Layer', value_name='Z-Score')
    
    # Map layer names for readability
    layer_map = {
        'edu_z': 'Education (Peer Cohort)',
        'emp_z': 'Employment (Brokerage)',
        'mem_z': 'Membership (Prestige)'
    }
    plot_data['Layer'] = plot_data['Layer'].map(layer_map)

    plt.figure(figsize=(12, 8))
    # Boxplot shows the distribution of scores for each role in each layer
    sns.boxplot(data=plot_data, x='Layer', y='Z-Score', hue='role', palette='Set2')
    
    plt.title('Structural Position of Roles Across Career Layers')
    plt.axhline(0, color='black', linestyle='--', linewidth=0.8) # Mean line
    plt.legend(title='Physicist Role', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Usage:
# visualize_role_structures(df_results)

In [ ]:
visualize_role_structures(results_dataframe)

### 1. What is a Z-Score?
A **Z-score** (or standard score) measures how many **standard deviations** a data point is away from the mean (average) of the group.

**Formula:**
$$Z = \frac{x - \mu}{\sigma}$$
*   $x$: The raw value (e.g., a physicist's Betweenness score).
*   $\mu$ (mu): The average of all scores in that graph.
*   $\sigma$ (sigma): The standard deviation (how spread out the scores are).

**Interpretation:**
*   **$Z = 0$**: The physicist is exactly average.
*   **$Z = +1.0$**: The physicist is **1 standard deviation above** the average (top ~16% in a normal distribution).
*   **$Z = -1.0$**: The physicist is **1 standard deviation below** the average.
*   **$Z = +2.5$**: The physicist is an extreme outlier (very high centrality).

---

### 2. Why is it critical in this specific analysis?

You are comparing three completely different mathematical metrics:
1.  **Degree Centrality (Education):** Values range from **0.0 to 1.0**. (e.g., Average might be 0.05).
2.  **Betweenness Centrality (Employment):** Values can range from **0.0 to 0.5** (or higher depending on graph size), but the distribution is usually very skewed. (e.g., Average might be 0.002).
3.  **PageRank/Eigenvector (Membership):** Values are relative weights that sum to 1. Individual scores are often tiny decimals like **0.0004**.

#### The Problem: "Apples vs. Oranges"
If you try to compare them directly without Z-scores:
*   Physicist A has Degree = **0.8** (Huge in Education).
*   Physicist A has Betweenness = **0.003** (Looks tiny in Employment).
*   Physicist A has PageRank = **0.0001** (Looks tiny in Membership).

If you sum these raw numbers, **Education dominates completely** because 0.8 is mathematically larger than 0.003. You would incorrectly conclude that Education is the most important factor, simply because its scale is larger.

#### The Solution: Z-Score Normalization
Z-scores convert all three metrics to a **common scale** based on *relative standing* within their own graph.

*   **Degree 0.8** might be $Z = +3.0$ (Top tier in Education).
*   **Betweenness 0.003** might *also* be $Z = +3.0$ (Top tier in Employment, even though the number looks small).
*   **PageRank 0.0001** might be $Z = +3.0$ (Top tier in Membership).

Now, you can fairly say: *"This physicist is in the top tier in ALL THREE layers,"* allowing you to correctly identify them as a **Star**.

---

### 3. How it Enables Role Classification
Your classification logic relies on thresholds like `HIGH = 0.8` and `LOW = -0.2`. These thresholds **only make sense** with Z-scores.

*   **Without Z-scores:** What is a "high" PageRank? Is 0.0005 high? Is 0.005 high? It depends entirely on the specific graph size and density. You would have to guess a new threshold for every dataset.
*   **With Z-scores:**
    *   `Z > 0.8` always means "significantly above average" (roughly top 20%).
    *   `Z < -0.2` always means "slightly below average".
    *   This makes your code **robust** and **portable**. You can run the same script on data from 1800 or 1980, and the definition of "Star" remains statistically consistent.


### Inspect profiles

In [ ]:
### Role in relation to country

crosstab = pd.crosstab(df_pa['role'],df_pa['country'] )

In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(8, 6))

In [ ]:
## role in relation to period
crosstab = pd.crosstab(df_pa['role'],df_pa['per_activ'] )


In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(8, 6))

In [ ]:
df_pa_filtered=df_pa[df_pa['role'] =='Academic Prince']
crosstab = pd.crosstab(df_pa_filtered['per_activ'],df_pa_filtered['country'] )

In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(8, 6))

#### Add roles to graphs

In [ ]:
### Prepare data to add to nodes
# The data must be of type : dictionary, i.e. key:value structure
lna = dict([(e['uri'],
     {'edu_z':e['edu_z'],
     'emp_z':e['emp_z'],
     'mem_z':e['mem_z'],
     'role':e['role']
     }
     ) for e in df_pa.to_dict(orient='records')])

In [ ]:
## Add attributes
nx.set_node_attributes(GA_edu, lna)
pprint.pprint(list(GA_edu.nodes.data())[:2])


In [ ]:
## Add attributes
nx.set_node_attributes(GA_empl, lna)
pprint.pprint(list(GA_empl.nodes.data())[:2])


In [ ]:
## Add attributes
nx.set_node_attributes(GA_memb, lna)
pprint.pprint(list(GA_memb.nodes.data())[:2])


## Centrality distributions

### Education

#### Degree centrality (edu)

In [ ]:
### Add degree centrality to nodes
degree = dict([(d[0], {'degree': d[1]}) for d in nx.degree(GA_edu)])
nx.set_node_attributes(GA_edu, degree)
pprint.pprint(list(GA_edu.nodes.data())[:2])

In [ ]:
### Statistical features of the degree
print(pd.Series([d[1] for d in nx.degree(GA_edu)]).describe())

In [ ]:
# Plot the distribution of degree centrality density

degree=[d[1] for d in nx.degree(GA_edu)]

min_val = min(degree)
max_val = max(degree)

plt.figure(figsize=(12, 2))
ax = sns.violinplot(data=degree, orient='h')

# Set the y-axis limits to exactly match the data range
ax.set_xlim(min_val, max_val)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.title('Distribution (density) of persons\' degree')
plt.show()

In [ ]:
### Add degree centrality to nodes
degree_centrality = nx.degree_centrality(GA_edu)
nx.set_node_attributes(GA_edu, degree_centrality, 'degree_centrality')
pprint.pprint(list(GA_edu.nodes.data())[10:12])


In [ ]:
### Graph to heavy to use Gephy online

# BEWARE : these can be very heavy graphs and shold not be committed to git!!!

nx.write_gexf(GA_edu, f"da_graphs/GA_edu.gexf")

### Inspect individuals

In [ ]:
### Export node attributes to dataframe
nodes_data ={node: GA_edu.nodes[node] for node in GA_edu.nodes}
nodes_df = pd.DataFrame(nodes_data).T
nodes_df.reset_index(inplace=True)
nodes_df.columns = ['personUri', 'label', 'birthYear', 'gender', 'per_activ', 'country', 
 'role','degree','degree_centrality', 'edu_z','emp_z','mem_z']
nodes_df.head(2)


#### Explore degree centrality on gr_edu

In [ ]:
nodes_df.sort_values(by='degree', ascending=False).iloc[30:40]

In [ ]:
duckdb.query("""
    SELECT label,degree, birthYear,country,per_activ,role,edu_z,emp_z, mem_z
    FROM nodes_df
    ORDER BY degree DESC
    LIMIT 10
""").to_df()

In [ ]:
duckdb.query("""
    SELECT label,degree, birthYear,country,per_activ,role,edu_z,emp_z, mem_z
    FROM nodes_df
    WHERE label LIKE '%Einstein%'
             or label LIKE '%Bohr%'
             or label LIKE '%Planck%'
    ORDER BY degree DESC
    LIMIT 10
""").to_df()

In [ ]:
gr_edu_degree_1000=nodes_df.sort_values(by='degree', ascending=False).iloc[:1000]
#gr_edu_degree_1000.head(2)
df=gr_edu_degree_1000
# Create the crosstab
crosstab = pd.crosstab(df['country'], df['per_activ'])


In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(8, 6))

In [ ]:
gr_edu_degree_1000.head(2)

### Get the most relevant educational institutions

In [ ]:
node_list = set(gr_edu_degree_1000.personUri)

edges_data = [
    {
        'source': u,
        'target': v,
        'orgsLabels': d.get('orgsLabels'),
        'orgsNumber': d.get('orgsNumber')

    } 
    for u, v, d in GA_edu.edges(data=True) 
    if u in node_list or v in node_list
]

df_edu_1000_edges = pd.DataFrame(edges_data, columns=['source', 'target', 'orgsLabels', 'orgsNumber'])

In [ ]:
df_edu_1000_edges[df_edu_1000_edges.orgsNumber>1].head()

In [ ]:
# Explode the pipe-separated values into separate rows
df_exploded = df_edu_1000_edges.assign(orgsLabels=df_edu_1000_edges['orgsLabels'].str.split('|')).explode('orgsLabels')
df_exploded.iloc[:3]


In [ ]:
df_exploded[df_exploded.orgsNumber>1].iloc[:3]

In [ ]:
# Strip whitespace and remove NaNs
df_exploded['orgsLabels'] = df_exploded['orgsLabels'].str.strip()
df_exploded = df_exploded.dropna(subset=['orgsLabels'])

# Count occurrences
university_counts = df_exploded['orgsLabels'].value_counts()

print(university_counts.iloc[:30])

### Slice graph using k-cores

Cf. this [notebook for the methodology](https://github.com/Sciences-historiques-numeriques/histoire_numerique_methodes/blob/main/analyse_reseaux/networkx_slicing_with_cores.ipynb)

In [ ]:
### Get k-cores

### A k-core is a maximal subgraph that contains nodes of degree k or more.
# https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.core.core_number.html#networkx.algorithms.core.core_number

core_numbers = nx.core_number(GA_edu)
print(str(core_numbers)[:150])


In [ ]:
### Distribution of nodes per core layer
l = [v for k,v in core_numbers.items()]
ls = pd.Series(l)
grouped = ls.groupby(ls).size()


ax = grouped.plot.bar(stacked=True, rot=70, fontsize=9, figsize=(10,8))
plt.title('Distribution of nodes per layers')

## https://stackoverflow.com/questions/71320380/add-only-the-total-values-on-top-of-stacked-bars
ax.bar_label(ax.containers[-1])

plt.show()

In [ ]:
# Identify layers based on core numbers
layers = {}
for node, core_number in core_numbers.items():
    if core_number not in layers:
        layers[core_number] = [node]
    else:
        layers[core_number].append(node)

In [ ]:
cts = []
for core_number, layer in layers.items():
    # print(f"layers {core_number}: {len(layer)}")
    cts.append([core_number,len(layer),layer])



cts=sorted(cts, key=lambda x: x[0])
p=[pprint.pprint([e[0],e[1],e[2][:1] ]) for e in cts]

nx.k_core: This function  performs an iterative peeling. If a node has degree 4 in the original graph but is connected to a node that gets removed, its degree drops. If it drops below 4, k_core removes it too. This continues until stability. This produces the true 4-core.

In [ ]:
### Same result by using the k_core function
kc_G = nx.k_core(GA_edu, 8)
naf.basic_graph_properties(kc_G)

In [ ]:
### Graph to heavy to use Gephy online

# BEWARE : these can be very heavy graphs and shold not be committed to git!!!

nx.write_gexf(kc_G, f"da_graphs/GA_edu_k8.gexf")

## Explore the employment relationships graph

### Centrality distributions

#### Eigenvector centrality

In [ ]:
### Add eigenvector to nodes

## If error: PowerIterationFailedConvergence: (PowerIterationFailedConvergence(...), 'power iteration failed to converge within 100 iterations')
# increase number of max iterations
eigenvector = nx.eigenvector_centrality(GA_empl, max_iter=200)
nx.set_node_attributes(GA_empl, eigenvector, 'eigenvector')
pprint.pprint(list(GA_empl.nodes.data())[:2])

In [ ]:
### Plot eigenvector density distribution 
eigenvector_s = pd.Series(list(eigenvector.values()))
stats = eigenvector_s.describe()

# Format decimal values
stats = stats.apply(lambda x: format(x, '.20f'))

print(stats)

In [ ]:
# Plot the distribution of eigenvector density

plt.figure(figsize=(8,4))
p = sns.violinplot(data=eigenvector_s[eigenvector_s> 0.00013610493515402787], orient='h')
plt.title('Eigenvector Distribution Density')
plt.show()

#### Betweenness centrality

In [ ]:
### Add betweenness to nodes
# Parameters for approximation: k=500, seed=42 and speed up

betweenness = nx.betweenness_centrality(GA_empl, k=500, seed=42)
nx.set_node_attributes(GA_empl, betweenness, 'betweenness')
pprint.pprint(list(GA_empl.nodes.data())[:2])

In [ ]:
# Plot the distribution of eigenbetweenness vector density

plt.figure(figsize=(8,4))
p = sns.violinplot(data=betweenness, orient='h')
plt.title('Betweenness Distribution Density')
plt.show()

In [ ]:
### Export node attributes to dataframe
nodes_data ={node: GA_empl.nodes[node] for node in GA_empl.nodes}
nodes_df_empl = pd.DataFrame(nodes_data).T
nodes_df_empl.reset_index(inplace=True)
nodes_df_empl.columns = ['personUri', 'label', 'birthYear', 'gender', 'per_activ', 'country', 
 'role', 'edu_z','emp_z','mem_z','eigenvector','betweenness']

In [ ]:
## Dataframe with the features of the nodes
nodes_df_empl.head(3)

#### Explore eigenvector on gr_empl

In [ ]:
nodes_df_empl.sort_values(by='eigenvector', ascending=False).iloc[:5]

In [ ]:
gr_empl_eigenv_1000=nodes_df_empl.sort_values(by='eigenvector', ascending=False).iloc[:1000]
gr_empl_eigenv_1000.head(2)

In [ ]:
df=gr_empl_eigenv_1000
# Create the crosstab
crosstab = pd.crosstab(df['country'], df['per_activ'])

In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(10, 6))

In [ ]:
crosstab

### Get the most relevant employment institutions

In [ ]:
node_list = set(gr_empl_eigenv_1000.personUri)

edges_data = [
    {
        'source': u,
        'target': v,
        'orgsLabels': d.get('orgsLabels'),
        'orgsNumber': d.get('orgsNumber')

    } 
    for u, v, d in GA_empl.edges(data=True) 
    if u in node_list or v in node_list
]

df_empl_eigenv_1000 = pd.DataFrame(edges_data, columns=['source', 'target', 'orgsLabels', 'orgsNumber'])

In [ ]:
# Explode the pipe-separated values into separate rows
df_exploded = df_empl_eigenv_1000.assign(orgsLabels=df_empl_eigenv_1000['orgsLabels'].str.split('|')).explode('orgsLabels')
# Strip whitespace and remove NaNs
df_exploded['orgsLabels'] = df_exploded['orgsLabels'].str.strip()
df_exploded = df_exploded.dropna(subset=['orgsLabels'])

# Count occurrences
university_counts = df_exploded['orgsLabels'].value_counts()

print(university_counts.iloc[:30])

#### Explore betweenness on gr_empl

In [ ]:
nodes_df_empl.sort_values(by='betweenness', ascending=False).iloc[:5]

In [ ]:
gr_empl_betweenness_1000=nodes_df_empl.sort_values(by='betweenness', ascending=False).iloc[:1000]
gr_empl_betweenness_1000.head(2)

In [ ]:
df=gr_empl_betweenness_1000
# Create the crosstab
crosstab = pd.crosstab(df['country'], df['per_activ'])


In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(10, 6))

### Get the most intermediary employment institutions

In [ ]:

node_list = set(gr_empl_betweenness_1000.personUri)

edges_data = [
    {
        'source': u,
        'target': v,
        'orgsLabels': d.get('orgsLabels'),
        'orgsNumber': d.get('orgsNumber')

    } 
    for u, v, d in GA_empl.edges(data=True) 
    if u in node_list or v in node_list
]

df_empl_eigenv_1000 = pd.DataFrame(edges_data, columns=['source', 'target', 'orgsLabels', 'orgsNumber'])

In [ ]:
# Explode the pipe-separated values into separate rows
df_exploded = df_empl_eigenv_1000.assign(orgsLabels=df_empl_eigenv_1000['orgsLabels'].str.split('|')).explode('orgsLabels')
# Strip whitespace and remove NaNs
df_exploded['orgsLabels'] = df_exploded['orgsLabels'].str.strip()
df_exploded = df_exploded.dropna(subset=['orgsLabels'])

# Count occurrences
university_counts = df_exploded['orgsLabels'].value_counts()

print(university_counts.iloc[:30])

### Create a subgraph with high eigenvector centrality scores

In [ ]:
centrality = {node: data.get('eigenvector', 0) for node, data in GA_empl.nodes(data=True)}
# Extract scores
scores = list(centrality.values())

In [ ]:
# Calculate third quartile (75th percentile)
threshold = np.percentile(scores, 75)

# Select nodes above the threshold
nodes_to_keep = [node for node, score in centrality.items() if score > threshold]

# Create subgraph
subGA_empl = GA_empl.subgraph(nodes_to_keep)
naf.basic_graph_properties(subGA_empl)

In [ ]:
### Graph to heavy to use Gephy online

# BEWARE : these can be very heavy graphs and shold not be committed to git!!!

nx.write_gexf(subGA_empl, f"da_graphs/subGA_empl.gexf")

## Explore the membership relationships graph

### Centrality distributions

#### Eigenvector centrality

In [ ]:
### Add eigenvector to nodes

## If error: PowerIterationFailedConvergence: (PowerIterationFailedConvergence(...), 'power iteration failed to converge within 100 iterations')
# increase number of max iterations
eigenvector = nx.eigenvector_centrality(GA_memb, max_iter=200)
nx.set_node_attributes(GA_memb, eigenvector, 'eigenvector')
pprint.pprint(list(GA_memb.nodes.data())[:2])

In [ ]:
# Plot the distribution of eigenvector density

plt.figure(figsize=(8,4))
p = sns.violinplot(data=eigenvector_s[eigenvector_s> 0.00013610493515402787], orient='h')
plt.title('Eigenvector Distribution Density')
plt.show()

In [ ]:
### Add betweenness to nodes
# Parameters for approximation: k=500, seed=42 and speed up

betweenness = nx.betweenness_centrality(GA_memb, k=500, seed=42)
nx.set_node_attributes(GA_memb, betweenness, 'betweenness')
pprint.pprint(list(GA_memb.nodes.data())[:2])

In [ ]:
# Plot the distribution of eigenbetweenness vector density
betweenness_s = pd.Series(list(betweenness.values()))

plt.figure(figsize=(8,4))
p = sns.violinplot(data=betweenness_s, orient='h')
plt.title('Betweenness Distribution Density')
plt.show()

In [ ]:
### Export node attributes to dataframe
nodes_data ={node: GA_memb.nodes[node] for node in GA_memb.nodes}
nodes_df_memb = pd.DataFrame(nodes_data).T
nodes_df_memb.reset_index(inplace=True)
nodes_df_memb.columns = ['personUri', 'label', 'birthYear', 'gender', 'per_activ', 'country', 
 'role', 'edu_z','emp_z','mem_z','eigenvector','betweenness']
nodes_df_memb.head(2)

#### Explore eigenvector on gr_memb

In [ ]:
nodes_df_memb.sort_values(by='eigenvector', ascending=False).iloc[:5]

In [ ]:
gr_memb_eigenv_1000=nodes_df_memb.sort_values(by='eigenvector', ascending=False).iloc[:1000]
gr_memb_eigenv_1000.head(2)

In [ ]:
### Most connected persons
df=gr_memb_eigenv_1000
# Create the crosstab
crosstab = pd.crosstab(df['country'], df['per_activ'])


In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
expected=bl.bivariate_stats(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(10, 4))

### Get the most relevant membership institutions

In [ ]:
node_list = set(gr_memb_eigenv_1000.personUri)

edges_data = [
    {
        'source': u,
        'target': v,
        'orgsLabels': d.get('orgsLabels'),
        'orgsNumber': d.get('orgsNumber')

    } 
    for u, v, d in GA_memb.edges(data=True) 
    if u in node_list or v in node_list
]

df_memb_eigenv_1000 = pd.DataFrame(edges_data, columns=['source', 'target', 'orgsLabels', 'orgsNumber'])

In [ ]:
# Explode the pipe-separated values into separate rows
df_exploded = df_memb_eigenv_1000.assign(orgsLabels=df_memb_eigenv_1000['orgsLabels'].str.split('|')).explode('orgsLabels')
# Strip whitespace and remove NaNs
df_exploded['orgsLabels'] = df_exploded['orgsLabels'].str.strip()
df_exploded = df_exploded.dropna(subset=['orgsLabels'])

# Count occurrences
university_counts = df_exploded['orgsLabels'].value_counts()

print(university_counts.iloc[:30])

In [ ]:
nodes_df_memb.sort_values(by='eigenvector', ascending=False).iloc[:10]

#### Explore betweenness on gr_memb

In [ ]:
nodes_df_memb.sort_values(by='betweenness', ascending=False).iloc[:5]

In [ ]:
gr_memb_betweenness_1000=nodes_df_memb.sort_values(by='betweenness', ascending=False).iloc[:1000]
gr_memb_betweenness_1000.head(2)

In [ ]:
df=gr_memb_betweenness_1000
# Create the crosstab
crosstab = pd.crosstab(df['country'], df['per_activ'])

In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(10, 6))

### Get the intermediary membership institutions

In [ ]:
node_list = set(gr_memb_betweenness_1000.personUri)

edges_data = [
    {
        'source': u,
        'target': v,
        'orgsLabels': d.get('orgsLabels'),
        'orgsNumber': d.get('orgsNumber')

    } 
    for u, v, d in GA_memb.edges(data=True) 
    if u in node_list or v in node_list
]

df_memb_betweenness_1000 = pd.DataFrame(edges_data, columns=['source', 'target', 'orgsLabels', 'orgsNumber'])

In [ ]:
# Explode the pipe-separated values into separate rows
df_exploded = df_memb_betweenness_1000.assign(orgsLabels=df_memb_betweenness_1000['orgsLabels'].str.split('|')).explode('orgsLabels')
# Strip whitespace and remove NaNs
df_exploded['orgsLabels'] = df_exploded['orgsLabels'].str.strip()
df_exploded = df_exploded.dropna(subset=['orgsLabels'])

# Count occurrences
university_counts = df_exploded['orgsLabels'].value_counts()

print(university_counts.iloc[:30])

In [ ]:
nodes_df_memb.sort_values(by='betweenness', ascending=False).iloc[:10]

### Produce subgraph

In [ ]:
df_exploded[:2]

In [ ]:
l_persons=duckdb.query("""
    SELECT DISTINCT source as person_uri
    FROM df_exploded
    WHERE orgsLabels IN ('French Academy of Sciences', 'Royal Swedish Academy of Sciences', 
                       'Göttingen Academy of Sciences and Humanities in Lower Saxony', 
                       'Bavarian Academy of Sciences and Humanities', 'Royal Netherlands Academy of Arts and Sciences',
                       'German Academy of Sciences Leopoldina')
    UNION
    SELECT target as person_uri
    FROM df_exploded
    WHERE orgsLabels IN ('French Academy of Sciences', 'Royal Swedish Academy of Sciences', 
                       'Göttingen Academy of Sciences and Humanities in Lower Saxony', 
                       'Bavarian Academy of Sciences and Humanities', 'Royal Netherlands Academy of Arts and Sciences',
                       'German Academy of Sciences Leopoldina ')

""").to_df()
print(len(l_persons))
l_persons.head(2)

In [ ]:
### Extract the subgraph using only the 1000 nodes
# .copy() creates a new independent graph object
sub_G = GA_memb.subgraph(l_persons.person_uri.to_list()).copy()

# 4. Verify and Export
print(f"Subgraph nodes: {sub_G.number_of_nodes()}")
print(f"Subgraph edges: {sub_G.number_of_edges()}")

graph_address="da_graphs/subgraph_membership_betweenness.graphml"

# Export directly to Gephi format
nx.write_graphml(sub_G, graph_address)